# Certificación Profesional en Desarrollador de Agentes de IA
## Clase 7 — RAG avanzado y debugging

ITBA Educación Ejecutiva

Diagnóstico y mejora del pipeline: el RAG de la Clase 6 funciona en el caso feliz — hoy encontramos dónde y por qué falla, y aplicamos técnicas concretas para mejorarlo.

### Cómo se usa este notebook

- El docente corre y explica cada celda en vivo. No programás en simultáneo durante la clase.
- Celdas **🏠 Practicá**: tarea individual para después de la clase.
- **Sección 10 (Laboratorio)**: checkpoint del proyecto integrador.

## Objetivo de la clase

Al final de la clase vas a poder:
- Clasificar una falla de RAG según la etapa del pipeline donde se originó (ingesta, recuperación, contexto, generación).
- Aplicar query rewriting y multi-query retrieval para mejorar recall en consultas ambiguas.
- Ajustar top-k con criterio, y detectar/eliminar redundancia de contexto.
- Detectar respuestas no sustentadas aunque la evidencia recuperada sea correcta.
- Armar un primer conjunto de casos de prueba para tu propio RAG.

**Checkpoint del proyecto integrador de hoy**: casos fallidos del RAG identificados, diagnosticados y corregidos (ver `proyecto-integrador/README.md`, checkpoint Clase 7).

## Agenda

1. Framework de diagnóstico por etapa
2. Errores de recuperación y query rewriting
3. Ajuste de top-k
4. Redundancia de contexto y context compression
5. Errores de generación: respuestas no sustentadas
6. Multi-query retrieval
7. Recuperación jerárquica
8. Síntesis: estrategias de debugging
9. Preparación de casos de prueba
10. Laboratorio: análisis y corrección de casos fallidos

## 0. Punto de partida

Reconstruimos la versión de la Clase 6 (`recuperar`, `armar_contexto`, `rag_con_registro`), con el mismo corpus de juguete — pero hoy le agregamos deliberadamente los problemas que vamos a diagnosticar: una consulta con vocabulario distinto al del corpus, un chunk casi redundante, y una pregunta ambigua.

In [ ]:
import os, json
from dotenv import load_dotenv
load_dotenv()

import boto3
bedrock = boto3.client("bedrock-runtime", region_name=os.environ.get("AWS_REGION", "us-east-1"))
MODEL_ID = os.environ.get("BEDROCK_MODEL_ID_CLAUDE", "global.anthropic.claude-sonnet-4-6")
EMBED_MODEL_ID = os.environ.get("BEDROCK_MODEL_ID_EMBED", "cohere.embed-v4:0")

def embed_bedrock(textos, input_type="search_document"):
    body = json.dumps({"texts": textos, "input_type": input_type, "embedding_types": ["float"]})
    resp = bedrock.invoke_model(modelId=EMBED_MODEL_ID, body=body, accept="*/*", contentType="application/json")
    return json.loads(resp["body"].read())["embeddings"]["float"]

def llamar_bedrock(prompt, system=None, temperature=0.0, max_tokens=500):
    kwargs = {"modelId": MODEL_ID, "messages": [{"role": "user", "content": [{"text": prompt}]}],
              "inferenceConfig": {"temperature": temperature, "maxTokens": max_tokens}}
    if system:
        kwargs["system"] = [{"text": system}]
    resp = bedrock.converse(**kwargs)
    return resp["output"]["message"]["content"][0]["text"]

In [ ]:
import chromadb

# Corpus de la Clase 6 + un chunk casi redundante (apis-003) agregado a propósito
corpus = [
    {"id": "apis-001", "texto": "La Converse API de Bedrock unifica la forma de llamar a distintos modelos: cambia el modelId, no el código.", "doc": "guia-apis"},
    {"id": "apis-002", "texto": "Los parámetros de inferencia como temperature y maxTokens van en inferenceConfig en Bedrock.", "doc": "guia-apis"},
    {"id": "apis-003", "texto": "La temperatura y el máximo de tokens se configuran dentro del campo inferenceConfig de Bedrock.", "doc": "guia-apis"},  # redundante con apis-002
    {"id": "seguridad-001", "texto": "La inyección de prompts ocurre cuando contenido no confiable manipula las instrucciones del modelo.", "doc": "guia-seguridad"},
    {"id": "seguridad-002", "texto": "El principio de mínimo privilegio limita qué herramientas puede invocar un agente según el contexto.", "doc": "guia-seguridad"},
    {"id": "rag-001", "texto": "Un pipeline RAG combina recuperación de evidencia con generación sustentada en esa evidencia.", "doc": "guia-rag"},
    {"id": "rag-002", "texto": "El top-k determina cuántos fragmentos recuperados se incluyen como contexto para el modelo generador.", "doc": "guia-rag"},
]

cliente_chroma = chromadb.Client()
coleccion = cliente_chroma.get_or_create_collection("corpus_clase7")
coleccion.add(
    ids=[c["id"] for c in corpus],
    embeddings=embed_bedrock([c["texto"] for c in corpus]),
    documents=[c["texto"] for c in corpus],
    metadatas=[{"doc": c["doc"]} for c in corpus],
)

def recuperar(consulta, k=3, where=None):
    vec = embed_bedrock([consulta], input_type="search_query")[0]
    kwargs = {"query_embeddings": [vec], "n_results": k}
    if where:
        kwargs["where"] = where
    resultado = coleccion.query(**kwargs)
    return [{"chunk_id": id_, "texto": t, "doc": m["doc"], "distancia": d}
            for id_, t, m, d in zip(resultado["ids"][0], resultado["documents"][0], resultado["metadatas"][0], resultado["distances"][0])]

print("Corpus listo:", coleccion.count(), "chunks")

## 1. Framework de diagnóstico por etapa

Cuando una respuesta de RAG está mal, el error puede venir de 4 lugares distintos — y la corrección es completamente distinta según cuál sea:

| Etapa | Pregunta diagnóstica | Si falla acá... |
|---|---|---|
| **Ingesta** (Clase 4) | ¿El chunk con la info correcta existe en el índice? | El chunking lo cortó mal, o el documento nunca se ingestó |
| **Recuperación** (Clase 5) | ¿El top-k trajo ese chunk? | El embedding no capturó bien la consulta, o k es muy chico |
| **Contexto** (Clase 6) | ¿El chunk recuperado llegó al prompt? | El umbral lo descartó, o se quedó sin presupuesto de tokens |
| **Generación** (Clase 6) | ¿El modelo usó el chunk que sí llegó al prompt? | El modelo ignoró evidencia válida (respuesta no sustentada) |

El `registro_consultas` de la Clase 6 (`chunks_recuperados` vs. `chunks_usados_en_contexto`) ya te da las primeras 3 respuestas sin tener que adivinar — hoy vamos a ver cómo usarlo.

In [ ]:
def diagnosticar(consulta, chunk_id_esperado, k=3):
    chunks = recuperar(consulta, k)
    ids_recuperados = [c["chunk_id"] for c in chunks]
    if chunk_id_esperado in ids_recuperados:
        print(f"✓ Recuperación OK: '{chunk_id_esperado}' está en el top-{k}.")
    else:
        print(f"✗ Falla de RECUPERACIÓN: '{chunk_id_esperado}' no está en el top-{k}. Recuperado: {ids_recuperados}")
    return chunks

# Consulta con vocabulario distinto al del corpus ("evitar que alguien le mienta al bot" en vez de "inyección de prompts")
diagnosticar("¿cómo evito que alguien le mienta al chatbot con instrucciones escondidas?", "seguridad-001", k=3)

## 2. Errores de recuperación y query rewriting

El embedding de una consulta coloquial puede no estar lo bastante cerca del embedding del chunk técnico correcto, aunque semánticamente apunten a lo mismo. **Query rewriting**: usar el propio LLM para reformular la consulta en el vocabulario del dominio *antes* de embeberla.

In [ ]:
SYSTEM_REWRITE = (
    "Reformulá la pregunta del usuario usando terminología técnica precisa de documentación de software "
    "(ej. 'inyección de prompts', 'ventana de contexto'). Devolvé SOLO la pregunta reformulada, sin explicación."
)

def reescribir_consulta(consulta: str) -> str:
    return llamar_bedrock(consulta, system=SYSTEM_REWRITE, max_tokens=60)


consulta_original = "¿cómo evito que alguien le mienta al chatbot con instrucciones escondidas?"
consulta_reescrita = reescribir_consulta(consulta_original)
print("Original: ", consulta_original)
print("Reescrita:", consulta_reescrita)

diagnosticar(consulta_reescrita, "seguridad-001", k=3)

**🏠 Practicá**: encontrá o inventá otra consulta coloquial sobre el corpus (ej. algo sobre "qué tan creativo es el modelo" para `apis-002`) y confirmá que el rewriting mejora la recuperación. Query rewriting tiene un costo: una llamada extra al modelo antes de cada búsqueda — en un sistema de alto volumen, hay que decidir si vale la pena siempre o solo cuando la recuperación directa falla.

## 3. Ajuste de top-k

No hay un k universal. k muy chico pierde evidencia relevante; k muy grande mete ruido (y gasta presupuesto de contexto en chunks poco relevantes).

In [ ]:
for k in [1, 3, 7]:
    chunks = recuperar("¿qué controla la temperatura de generación?", k=min(k, coleccion.count()))
    distancias = [round(c["distancia"], 3) for c in chunks]
    print(f"k={k}: {len(chunks)} chunks, distancias={distancias}")

Con k=7 (todo el corpus) ves la distancia subir mucho hacia el final de la lista — esos últimos chunks probablemente no deberían llegar al contexto. Esto es exactamente lo que el umbral de la Clase 6 filtra; ajustar k y ajustar el umbral son dos palancas relacionadas pero distintas: k limita *cuántos* candidatos se evalúan, el umbral decide *cuáles* de esos candidatos son lo bastante buenos.

## 4. Redundancia de contexto y context compression

Con `apis-002` y `apis-003` (casi idénticos) en el corpus, el top-k puede gastar dos lugares en decir lo mismo dos veces, desplazando un chunk distinto pero relevante.

In [ ]:
chunks_con_redundancia = recuperar("¿qué controla la temperatura de generación?", k=3)
for c in chunks_con_redundancia:
    print(f"{c['chunk_id']}  (dist={c['distancia']:.3f})  {c['texto'][:60]}...")

In [ ]:
def deduplicar_por_similitud(chunks: list[dict], umbral_similitud: float = 0.92) -> list[dict]:
    """Descarta chunks cuyo contenido es casi idéntico a uno ya aceptado (comparando embeddings entre sí)."""
    if not chunks:
        return chunks
    vectores = embed_bedrock([c["texto"] for c in chunks], input_type="search_document")
    aceptados, vectores_aceptados = [chunks[0]], [vectores[0]]
    for c, v in zip(chunks[1:], vectores[1:]):
        es_duplicado = any(
            (sum(a*b for a, b in zip(v, va)) / ((sum(x*x for x in v)**0.5) * (sum(x*x for x in va)**0.5))) > umbral_similitud
            for va in vectores_aceptados
        )
        if not es_duplicado:
            aceptados.append(c)
            vectores_aceptados.append(v)
        else:
            print(f"Descartado por redundante: {c['chunk_id']}")
    return aceptados


chunks_deduplicados = deduplicar_por_similitud(chunks_con_redundancia)
print("\nChunks finales:", [c["chunk_id"] for c in chunks_deduplicados])

**Context compression** (concepto, sin implementarlo hoy en código): en vez de solo deduplicar chunks completos, se puede comprimir cada chunk a la oración o frase más relevante para la consulta específica (con otra llamada al LLM, o con un modelo de extracción más chico) — así un chunk largo que solo tiene una frase relevante no gasta todo su presupuesto de tokens en texto irrelevante. Es una optimización de costo/densidad, no de corrección — aplicarla cuando el presupuesto de contexto (Clase 6) sea la limitación real, no antes.

## 5. Errores de generación: respuestas no sustentadas

El caso más insidioso: la recuperación y el contexto están bien, pero el modelo **igual** no usa la evidencia correctamente — ignora un chunk, mezcla información de dos chunks de forma incorrecta, o cita mal.

In [ ]:
import re

def validar_citas(respuesta: str, cantidad_fuentes: int) -> list[int]:
    """Devuelve los números de cita citados en la respuesta que NO corresponden a ninguna fuente real."""
    citas = {int(n) for n in re.findall(r"\[(\d+)\]", respuesta)}
    return sorted(n for n in citas if n < 1 or n > cantidad_fuentes)


respuesta_con_cita_invalida = "La temperatura se configura en inferenceConfig [1]. También soporta streaming [5]."
print("Citas inválidas detectadas:", validar_citas(respuesta_con_cita_invalida, cantidad_fuentes=3))

Esta validación atrapa citas a fuentes que no existen — pero no atrapa el caso más común (y más difícil) de respuesta no sustentada: el modelo cita correctamente `[1]`, pero lo que dice **no es lo que dice el chunk 1**. Para eso no hay una validación automática simple; hace falta revisión humana sobre una muestra, o (Clase 8) un juez automático (LLM-as-a-Judge) que compare la afirmación contra la evidencia citada.

**🏠 Practicá**: armá un caso donde el chunk 1 diga una cosa y la "respuesta del modelo" (escribila vos a mano, simulando una alucinación) diga algo distinto pero cite `[1]` igual. Confirmá que `validar_citas` no lo detecta — es la prueba de que esta validación es necesaria pero no suficiente.

## 6. Multi-query retrieval

En vez de una sola reformulación (sección 2), generar **varias** variantes de la consulta, recuperar para cada una, y combinar — mejora el *recall* cuando la pregunta original es ambigua o admite más de una interpretación razonable.

In [ ]:
SYSTEM_MULTIQUERY = (
    "Generá 3 reformulaciones distintas de la pregunta del usuario, cada una enfatizando un aspecto o "
    "interpretación posible. Una por línea, sin numerar ni explicar."
)

def generar_variantes(consulta: str, n: int = 3) -> list[str]:
    respuesta = llamar_bedrock(consulta, system=SYSTEM_MULTIQUERY, max_tokens=150)
    variantes = [l.strip("- ").strip() for l in respuesta.split("\n") if l.strip()]
    return variantes[:n]


def recuperar_multiquery(consulta: str, k_por_variante: int = 2) -> list[dict]:
    variantes = generar_variantes(consulta) + [consulta]
    vistos, combinados = set(), []
    for v in variantes:
        for c in recuperar(v, k=k_por_variante):
            if c["chunk_id"] not in vistos:
                vistos.add(c["chunk_id"])
                combinados.append(c)
    return sorted(combinados, key=lambda c: c["distancia"])


resultado_multiquery = recuperar_multiquery("¿cómo se controla el comportamiento del modelo al generar texto?")
for c in resultado_multiquery:
    print(f"{c['chunk_id']}  (dist={c['distancia']:.3f})")

**🏠 Practicá**: compará `recuperar_multiquery` contra `recuperar` simple (k=3) para la misma consulta — ¿multi-query trae algún chunk relevante que la búsqueda simple se perdía? Tiene el mismo trade-off que query rewriting: más llamadas al modelo, más costo y latencia, a cambio de mejor recall.

## 7. Recuperación jerárquica

Para corpus grandes (el real del TP, MDN Web Docs, tiene ~28 subtemas — mucho más que nuestro corpus de juguete), buscar directo entre miles de chunks puede traer resultados de un documento totalmente distinto al que realmente responde la pregunta. **Recuperación jerárquica**: primero encontrar el *documento* correcto (nivel alto), después buscar *dentro* de ese documento (nivel bajo) — usando el filtro de metadata que ya conocés de la Clase 5.

In [ ]:
def recuperar_jerarquico(consulta: str, k_docs: int = 1, k_chunks: int = 3) -> list[dict]:
    # Paso 1: ¿qué documento(s) son más relevantes? (acá, con pocos docs, alcanza con mirar el doc del chunk más cercano)
    candidatos = recuperar(consulta, k=1)
    docs_relevantes = list({c["doc"] for c in candidatos[:k_docs]})

    # Paso 2: buscar solo dentro de esos documentos
    return recuperar(consulta, k=k_chunks, where={"doc": {"$in": docs_relevantes}})


resultado_jerarquico = recuperar_jerarquico("¿qué es el mínimo privilegio?")
for c in resultado_jerarquico:
    print(f"{c['chunk_id']} (doc={c['doc']})")

Con nuestro corpus de 7 chunks esto es overkill — la ganancia real aparece con corpus grandes, donde filtrar primero por documento reduce drásticamente el espacio de búsqueda del paso 2 y evita que un chunk de otro documento, casualmente parecido en superficie, se cuele en el top-k.

## 8. Síntesis: estrategias de debugging

Ante una respuesta de RAG que "está mal", en orden:

1. **Localizá la etapa** con el framework de la sección 1 (¿existe el chunk? ¿se recuperó? ¿llegó al contexto? ¿se usó bien?).
2. Si es **ingesta**: revisá el chunking de ese documento específico (Clase 4) — ¿el chunk quedó cortado a la mitad de la idea?
3. Si es **recuperación**: probá query rewriting (sección 2) o multi-query (sección 6) antes de asumir que "el embedding no sirve".
4. Si es **contexto**: revisá umbral y presupuesto de tokens (Clase 6) — ¿se está descartando algo que sí llegó en el top-k?
5. Si es **generación**: revisá el prompt y la temperatura — y validá citas (sección 5) como primera línea de defensa, sabiendo que no es suficiente por sí sola.

No optimices una etapa que no es donde está el problema — es la forma más común de perder tiempo en RAG (ej. subir k cuando el problema real era que el chunk nunca se ingestó).

## 9. Preparación de casos de prueba

Antes de la Clase 8 (evaluación formal con métricas), conviene tener ya un set de preguntas de prueba con una noción de "qué se espera" — la base de cualquier dataset de evaluación.

In [ ]:
casos_de_prueba = [
    {"pregunta": "¿Qué parámetro controla la temperatura de generación?", "chunk_esperado": "apis-002", "debe_responder": True},
    {"pregunta": "¿Qué es la inyección de prompts?", "chunk_esperado": "seguridad-001", "debe_responder": True},
    {"pregunta": "¿Cómo se hace un backup de la base de datos?", "chunk_esperado": None, "debe_responder": False},
]

for caso in casos_de_prueba:
    chunks = recuperar(caso["pregunta"], k=3)
    recuperado_ok = caso["chunk_esperado"] in [c["chunk_id"] for c in chunks] if caso["chunk_esperado"] else True
    estado = "✓" if recuperado_ok == caso["debe_responder"] or (caso["chunk_esperado"] and recuperado_ok) else "✗"
    print(f"{estado} {caso['pregunta']}")

**🏠 Practicá**: agregá 3 casos de prueba más sobre tu propio corpus del TP, incluyendo al menos un caso ambiguo (una pregunta que podría responderse con más de un chunk) — guardalos, los vas a reutilizar como base del dataset de evaluación de la Clase 8.

## 10. Laboratorio: análisis y corrección de casos fallidos

**🏠 Tarea de la semana — entregar/completar antes de la Clase 8.**

**Checkpoint del proyecto integrador (Clase 7)**: casos fallidos del RAG real (el de tu TP, no el de juguete de hoy) identificados, diagnosticados por etapa, y corregidos con al menos una de las técnicas de hoy.

### Consigna

1. Corré tu pipeline RAG de la Clase 6 sobre al menos 8-10 preguntas reales de tu corpus (MDN Web Docs) — algunas directas, algunas con vocabulario coloquial, alguna fuera de dominio.
2. Para cada caso que falle (respuesta incorrecta, no sustentada, o "no sé" cuando sí había evidencia), usá el framework de la sección 1 para clasificar en qué etapa está el problema.
3. Aplicá la técnica correspondiente de esta clase (query rewriting, ajuste de k, deduplicación, multi-query, o recuperación jerárquica) a **al menos 2 casos distintos**, y confirmá que la corrección efectivamente mejora el resultado.
4. Documentá en una celda markdown: qué caso falló, por qué (según el diagnóstico), qué técnica aplicaste, y el resultado antes/después.

### Criterio de éxito

- Cada caso corregido tiene un diagnóstico explícito de etapa, no solo "lo arreglé".
- Al menos 2 técnicas distintas de esta clase están aplicadas y comparadas con un antes/después concreto (no solo descriptas).
- Tenés un primer conjunto de casos de prueba (sección 9) con al menos 8 preguntas, listo para la Clase 8.

In [ ]:
def diagnosticar_y_corregir(consulta: str, pipeline_rag) -> dict:
    # TODO: correr pipeline_rag(consulta) y guardar el resultado "antes"
    # TODO: clasificar la falla según el framework de la sección 1 usando el registro de la Clase 6
    # TODO: aplicar la corrección que corresponda (reescribir_consulta, ajustar k, deduplicar, multiquery, jerárquico)
    # TODO: volver a correr y devolver {"antes":, "despues":, "diagnostico":, "tecnica_aplicada":}
    raise NotImplementedError


# Prueba (va a fallar hasta que completes el TODO):
# print(diagnosticar_y_corregir("una pregunta que sabés que falla", tu_pipeline_rag_de_la_clase_6))

## Cierre

- Hoy el RAG dejó de ser "funciona o no funciona" y se volvió diagnosticable: 4 etapas, cada una con sus propios síntomas y correcciones.
- Las técnicas de hoy (rewriting, multi-query, deduplicación, recuperación jerárquica) no son gratis — todas cuestan latencia, costo, o complejidad. Aplicarlas donde el diagnóstico las justifica, no por defecto en todos los casos.
- Próxima clase: evaluación — en vez de diagnosticar caso por caso a mano, vamos a medir con métricas sobre el conjunto de casos de prueba que empezaste hoy.
- Antes de la próxima clase: dejar el laboratorio de la sección 10 funcionando, con al menos 2 casos corregidos y documentados.